# Tugas Take-Home Sesi 05 — Transfer Learning untuk Klasifikasi Citra dengan Data Terbatas
**Dataset:** Oxford-IIIT Pet (subset 10 kelas, split stratified 70/15/15) | **Backbone utama:** ResNet-18 | **Seed:** 42

**Cara pakai:** Colab → Runtime → Change runtime type → **T4 GPU** → *Run all*. Semua metrik tersimpan otomatis di folder `outputs/` (`results.csv`, `results.json`, `history_all.csv`, `run_config.json`, gambar, checkpoint). Semua angka di laporan harus diambil dari berkas-berkas tersebut.

**Keputusan desain (dapat diubah di sel konfigurasi):**
1. B1, B2, B3 dilatih **tanpa augmentasi** agar B4 menjadi uji kontrol yang bersih (B4 = kondisi `B4_BASE` dengan augmentasi, dibandingkan dengan versi tanpa augmentasinya).
2. Layer BatchNorm pada bagian backbone yang dibekukan dipaksa `eval()` agar running statistics ImageNet tidak berubah (lihat `set_train_mode`).
3. Model terbaik dipilih berdasarkan **validation**, bukan test, supaya tidak terjadi kebocoran seleksi ke test set.

In [ ]:
import os, random, time, json, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from PIL import Image

# ---------------- Konfigurasi ----------------
SEED = 42
BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5            # early stopping pada validation loss (3-5)
WEIGHT_DECAY = 1e-2
NUM_WORKERS = 2
LR_SCRATCH = 1e-3       # B1: seluruh parameter
LR_HEAD = 1e-3          # fc / classifier head baru
LR_BLOCK = 1e-4         # layer4 (B3)
B4_BASE = "B3"          # "B2" atau "B3": kondisi yang diuji dengan/tanpa augmentasi
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASSES = ["Abyssinian", "Bengal", "Birman", "Persian", "Siamese",          # 5 kucing
           "Beagle", "Pug", "Samoyed", "Shiba Inu", "Chihuahua"]            # 5 anjing
NUM_CLASSES = len(CLASSES)

ROOT, OUT = Path("data"), Path("outputs")
CKPT, FIG = OUT / "checkpoints", OUT / "figures"
for d in (ROOT, CKPT, FIG): d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    s = SEED + worker_id
    np.random.seed(s); random.seed(s)

def sync():
    if device.type == "cuda": torch.cuda.synchronize()

set_seed()
run_config = dict(seed=SEED, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                  weight_decay=WEIGHT_DECAY, lr_scratch=LR_SCRATCH, lr_head=LR_HEAD, lr_block=LR_BLOCK,
                  b4_base=B4_BASE, classes=CLASSES, device=str(device),
                  gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                  torch=torch.__version__, torchvision=torchvision.__version__, python=platform.python_version())
json.dump(run_config, open(OUT / "run_config.json", "w"), indent=2)
print(run_config)

## 1. Data: subset 10 kelas + split stratified 70/15/15
Split dilakukan **sekali** pada level indeks gambar (bukan setelah transformasi), sehingga tidak ada kebocoran antar-split.

In [ ]:
raw_tv = datasets.OxfordIIITPet(root=ROOT, split="trainval", target_types="category", download=True)
raw_te = datasets.OxfordIIITPet(root=ROOT, split="test", target_types="category", download=True)

all_classes = raw_tv.classes
missing = [c for c in CLASSES if c not in all_classes]
assert not missing, f"Nama kelas tidak ditemukan: {missing}\nKelas tersedia: {all_classes}"
cls_map = {all_classes.index(c): i for i, c in enumerate(CLASSES)}

paths, labels = [], []
for ds in (raw_tv, raw_te):                      # gabungkan lalu split ulang sendiri
    for p, l in zip(ds._images, ds._labels):
        if l in cls_map:
            paths.append(str(p)); labels.append(cls_map[l])
paths, labels = np.array(paths), np.array(labels)

idx = np.arange(len(paths))
train_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=labels, random_state=SEED)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.50, stratify=labels[tmp_idx], random_state=SEED)
SPLITS = {"train": train_idx, "val": val_idx, "test": test_idx}

# Cek kebocoran antar-split
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))
print(f"Total: {len(paths)} | train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")
dist = pd.DataFrame({s: np.bincount(labels[i], minlength=NUM_CLASSES) for s, i in SPLITS.items()}, index=CLASSES)
display(dist)
pd.DataFrame({"path": paths, "label": labels, "split": [next(s for s, i in SPLITS.items() if k in set(i)) for k in idx]}
             ).to_csv(OUT / "split_assignment.csv", index=False)

In [ ]:
tf_plain = transforms.Compose([                       # train tanpa augmentasi + val/test
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

tf_aug = transforms.Compose([                         # hanya untuk train (sesuai starter code)
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class PetDataset(Dataset):
    def __init__(self, paths, labels, tf): self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), int(self.labels[i])

def make_loader(split, aug=False, shuffle=False):
    ii = SPLITS[split]
    tf = tf_aug if (split == "train" and aug) else tf_plain   # augmentasi TIDAK pernah aktif di val/test
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(PetDataset(paths[ii], labels[ii], tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      drop_last=(split == "train" and shuffle), num_workers=NUM_WORKERS,
                      worker_init_fn=seed_worker, generator=g, pin_memory=(device.type == "cuda"))

xb, yb = next(iter(make_loader("train", aug=True, shuffle=True)))
print("Batch:", xb.shape, yb.shape)   # [32, 3, 224, 224] [32]

## 2. Model builder, strategi freezing, dan pencatatan parameter
- `scratch` (B1): `weights=None`, semua parameter dilatih.
- `fe` (B2): backbone beku, hanya head baru yang dilatih.
- `ft` (B3): `layer4` + head dibuka, **differential learning rate** (1e-4 vs 1e-3).

**Pitfall penting:** `requires_grad=False` tidak menghentikan BatchNorm memperbarui running mean/var saat `model.train()`. Karena itu BN pada bagian beku dipaksa `eval()`.

In [ ]:
def count_params(m):
    return (sum(p.numel() for p in m.parameters() if p.requires_grad),
            sum(p.numel() for p in m.parameters()))

def build_model(arch, pretrained, strategy):
    W = {"resnet18": models.ResNet18_Weights.IMAGENET1K_V1,
         "efficientnet_b0": models.EfficientNet_B0_Weights.IMAGENET1K_V1,
         "mobilenet_v3_small": models.MobileNet_V3_Small_Weights.IMAGENET1K_V1}[arch]
    model = getattr(models, arch)(weights=W if pretrained else None)

    if arch == "resnet18":
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        head, last_block = model.fc, model.layer4
    elif arch == "efficientnet_b0":
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
        head, last_block = model.classifier[1], nn.ModuleList(list(model.features)[-2:])
    else:  # mobilenet_v3_small
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
        head, last_block = model.classifier[3], nn.ModuleList(list(model.features)[-3:])

    if strategy == "scratch":
        groups = [{"name": "all", "params": list(model.parameters()), "lr": LR_SCRATCH}]
    else:
        for p in model.parameters(): p.requires_grad = False
        for p in head.parameters(): p.requires_grad = True
        groups = [{"name": "head", "params": list(head.parameters()), "lr": LR_HEAD}]
        if strategy == "ft":
            for p in last_block.parameters(): p.requires_grad = True
            groups.insert(0, {"name": "last_block", "params": list(last_block.parameters()), "lr": LR_BLOCK})
    return model, groups

def set_train_mode(model):
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()                      # BN beku -> pakai running stats pretrained

# Sanity check sesuai Tips Implementasi (harus 5.130 untuk B2 pada ResNet-18)
_m, _ = build_model("resnet18", True, "fe")
tp, tot = count_params(_m); print(f"B2 trainable: {tp:,} / {tot:,}"); assert tp == 5130
_m, _ = build_model("resnet18", True, "ft"); print("B3 trainable: %s / %s" % tuple(f"{v:,}" for v in count_params(_m)))
del _m

## 3. Training loop, evaluasi, early stopping, dan logging
Metrik yang dicatat per eksperimen: trainable params, LR per grup, epoch konvergen (epoch dengan val loss terbaik), durasi total, val accuracy, Macro-F1 test. Ditambah `train_acc_clean`: akurasi train pada model terbaik **tanpa augmentasi**, supaya generalization gap (train − val) adil dibandingkan antar kondisi.

In [ ]:
RESULTS, HISTORY, PREDS, CONFIGS = {}, [], {}, {}

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ls, Y, PR = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        ls += F.cross_entropy(out, y, reduction="sum").item()
        Y.append(y.cpu().numpy()); PR.append(out.softmax(1).cpu().numpy())
    y, prob = np.concatenate(Y), np.concatenate(PR); pred = prob.argmax(1)
    return dict(loss=ls / len(y), acc=float((pred == y).mean()),
                macro_f1=float(f1_score(y, pred, average="macro")), y=y, pred=pred, prob=prob)

def train_one_epoch(model, loader, optimizer):
    set_train_mode(model); ls, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        out = model(x); loss = F.cross_entropy(out, y)
        loss.backward(); optimizer.step()
        ls += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return ls / n, correct / n

def persist():
    pd.DataFrame(list(RESULTS.values())).to_csv(OUT / "results.csv", index=False)
    json.dump(list(RESULTS.values()), open(OUT / "results.json", "w"), indent=2)
    pd.DataFrame(HISTORY).to_csv(OUT / "history_all.csv", index=False)

def run_experiment(exp_id, arch, strategy, aug, pretrained=None):
    pretrained = (strategy != "scratch") if pretrained is None else pretrained
    print(f"\n{'='*72}\n{exp_id} | arch={arch} | strategy={strategy} | augmentation={aug}\n{'='*72}")
    set_seed(SEED)
    model, groups = build_model(arch, pretrained, strategy); model.to(device)
    trainable, total = count_params(model)                 # verifikasi SEBELUM optimizer dibentuk
    print(f"Trainable Parameters: {trainable:,} / {total:,}")
    optimizer = torch.optim.AdamW([{"params": g["params"], "lr": g["lr"]} for g in groups], weight_decay=WEIGHT_DECAY)
    tr, va, te, tr_clean = make_loader("train", aug, True), make_loader("val"), make_loader("test"), make_loader("train")

    best, bad = {"loss": float("inf"), "epoch": 0, "state": None}, 0
    HISTORY[:] = [h for h in HISTORY if h["exp"] != exp_id]
    sync(); t0 = time.time(); ep = 0
    for ep in range(1, MAX_EPOCHS + 1):
        te0 = time.time()
        trl, tra = train_one_epoch(model, tr, optimizer)
        v = evaluate(model, va)
        HISTORY.append(dict(exp=exp_id, epoch=ep, train_loss=trl, train_acc=tra,
                            val_loss=v["loss"], val_acc=v["acc"], epoch_sec=time.time() - te0))
        print(f"ep {ep:02d} | train loss {trl:.4f} acc {tra:.3f} | val loss {v['loss']:.4f} acc {v['acc']:.3f}")
        if v["loss"] < best["loss"] - 1e-6:
            best = {"loss": v["loss"], "epoch": ep, "state": {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}}; bad = 0
        else:
            bad += 1
            if bad >= PATIENCE: print(f"Early stopping di epoch {ep}"); break
    sync(); train_time = time.time() - t0

    model.load_state_dict(best["state"])
    v, t, trc = evaluate(model, va), evaluate(model, te), evaluate(model, tr_clean)
    torch.save(best["state"], CKPT / f"{exp_id}.pt")
    PREDS[exp_id] = dict(y=t["y"], pred=t["pred"], prob=t["prob"])
    np.savez(OUT / f"predictions_{exp_id}.npz", **PREDS[exp_id])
    CONFIGS[exp_id] = dict(arch=arch, strategy=strategy, aug=aug, pretrained=pretrained)

    RESULTS[exp_id] = dict(exp=exp_id, arch=arch, strategy=strategy, pretrained=pretrained, augmentation=aug,
        trainable_params=trainable, total_params=total, lr_groups=json.dumps({g["name"]: g["lr"] for g in groups}),
        epochs_run=ep, best_epoch=best["epoch"], train_time_sec=round(train_time, 1),
        sec_per_epoch=round(train_time / ep, 2), val_loss=v["loss"], val_acc=v["acc"], val_macro_f1=v["macro_f1"],
        train_acc_clean=trc["acc"], gap_train_val=trc["acc"] - v["acc"],
        test_acc=t["acc"], test_macro_f1=t["macro_f1"], seed=SEED, batch_size=BATCH_SIZE)
    persist()
    r = RESULTS[exp_id]
    print(f"--> best epoch {r['best_epoch']} | val acc {r['val_acc']:.4f} | test Macro-F1 {r['test_macro_f1']:.4f} | "
          f"gap {r['gap_train_val']:.3f} | {r['train_time_sec']}s")
    return r

## 4. Eksperimen B1–B3 (tanpa augmentasi)

In [ ]:
run_experiment("B1", "resnet18", "scratch", aug=False)   # baseline from scratch
run_experiment("B2", "resnet18", "fe",      aug=False)   # feature extraction
run_experiment("B3", "resnet18", "ft",      aug=False)   # fine-tuning layer4 + fc, differential LR

## 5. B4 — Uji kontrol augmentasi
Kondisi `B4_BASE` tanpa augmentasi sudah dijalankan di atas; di sini dijalankan versi dengan augmentasi (seed, protokol, dan hyperparameter identik).

In [ ]:
base = CONFIGS[B4_BASE]
B4_ID = f"B4_{B4_BASE}_aug"
run_experiment(B4_ID, base["arch"], base["strategy"], aug=True)
d = pd.DataFrame([RESULTS[B4_BASE], RESULTS[B4_ID]])[["exp", "augmentation", "train_acc_clean", "val_acc", "gap_train_val", "val_loss", "test_macro_f1"]]
display(d)

## 6. B5 (Bonus) — Arsitektur efisien
Konfigurasi terbaik (dipilih dari **validation**: B2/B3/B4) diterapkan pada EfficientNet-B0 dan MobileNetV3-Small.

In [ ]:
cands = [k for k in RESULTS if k in ("B2", "B3") or k.startswith("B4")]
best_cfg_id = max(cands, key=lambda k: (RESULTS[k]["val_acc"], -RESULTS[k]["val_loss"]))
cfg = CONFIGS[best_cfg_id]
print(f"Konfigurasi terbaik (val): {best_cfg_id} -> strategy={cfg['strategy']}, aug={cfg['aug']}")
for arch in ("efficientnet_b0", "mobilenet_v3_small"):
    run_experiment(f"B5_{arch}", arch, cfg["strategy"], aug=cfg["aug"])

## 7. C1 — Tabel konsolidasi dan kurva pembelajaran

In [ ]:
cols = ["exp", "strategy", "arch", "augmentation", "trainable_params", "total_params", "lr_groups", "best_epoch",
        "epochs_run", "train_time_sec", "val_acc", "train_acc_clean", "gap_train_val", "test_acc", "test_macro_f1"]
table = pd.DataFrame(list(RESULTS.values()))[cols]
table.to_csv(OUT / "table_consolidated.csv", index=False)
display(table.style.format({"val_acc": "{:.4f}", "train_acc_clean": "{:.4f}", "gap_train_val": "{:.4f}",
                            "test_acc": "{:.4f}", "test_macro_f1": "{:.4f}", "trainable_params": "{:,}", "total_params": "{:,}"}))

H = pd.DataFrame(HISTORY)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, e in zip(axes, ["B1", "B2", "B3"]):
    h = H[H.exp == e]
    ax.plot(h.epoch, h.train_loss, label="train loss"); ax.plot(h.epoch, h.val_loss, label="val loss")
    ax.axvline(RESULTS[e]["best_epoch"], ls="--", c="gray", label="best epoch")
    ax.set_title(f"{e} ({RESULTS[e]['strategy']})"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "C1_loss_curves_B1_B2_B3.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, e in zip(axes, [B4_BASE, B4_ID]):
    h = H[H.exp == e]
    ax.plot(h.epoch, h.train_acc, label="train acc (running)"); ax.plot(h.epoch, h.val_acc, label="val acc")
    ax.set_title(f"B4: {e}"); ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "B4_augmentation_curves.png", dpi=200); plt.show()

## 8. C3 — Failure analysis (model ResNet-18 terbaik menurut validation)
Sampel yang ditampilkan adalah kesalahan dengan **confidence tertinggi** (paling informatif). Kolom `hipotesis_penyebab` di CSV sengaja dikosongkan: isi sendiri setelah melihat gambarnya.

In [ ]:
r18 = [k for k in RESULTS if RESULTS[k]["arch"] == "resnet18"]
best_id = max(r18, key=lambda k: (RESULTS[k]["val_acc"], -RESULTS[k]["val_loss"]))
P = PREDS[best_id]; test_paths = paths[SPLITS["test"]]
print("Model terbaik:", best_id, "| test acc", round(RESULTS[best_id]["test_acc"], 4), "| Macro-F1", round(RESULTS[best_id]["test_macro_f1"], 4))
print(classification_report(P["y"], P["pred"], target_names=CLASSES, digits=3))

wrong = np.where(P["pred"] != P["y"])[0]
conf = P["prob"][wrong, P["pred"][wrong]]
order = wrong[np.argsort(-conf)]
top2 = np.argsort(-P["prob"], axis=1)[:, :2]
rows = [dict(test_idx=int(i), path=test_paths[i], ground_truth=CLASSES[P["y"][i]], predicted=CLASSES[P["pred"][i]],
             confidence=float(P["prob"][i, P["pred"][i]]), second_choice=CLASSES[top2[i, 1]] if top2[i, 0] == P["pred"][i] else CLASSES[top2[i, 0]],
             hipotesis_penyebab="") for i in order]
pd.DataFrame(rows).to_csv(OUT / "failure_analysis.csv", index=False)
print(f"Total salah prediksi di test set: {len(wrong)} / {len(P['y'])}")

show = order[:8]
if len(show):
    cols_n = 4; rows_n = int(np.ceil(len(show) / cols_n))
    fig, axes = plt.subplots(rows_n, cols_n, figsize=(4 * cols_n, 4.3 * rows_n)); axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, i in zip(axes, show):
        ax.imshow(Image.open(test_paths[i]).convert("RGB"))
        ax.set_title(f"GT: {CLASSES[P['y'][i]]}\nPred: {CLASSES[P['pred'][i]]} ({P['prob'][i, P['pred'][i]]:.2f})", fontsize=10)
    plt.tight_layout(); plt.savefig(FIG / "C3_failure_samples.png", dpi=200); plt.show()

cm = confusion_matrix(P["y"], P["pred"])
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xticks(range(NUM_CLASSES), CLASSES, rotation=60, ha="right"); plt.yticks(range(NUM_CLASSES), CLASSES)
for a in range(NUM_CLASSES):
    for b in range(NUM_CLASSES): plt.text(b, a, cm[a, b], ha="center", va="center", fontsize=8)
plt.xlabel("Predicted"); plt.ylabel("Ground truth"); plt.title(f"Confusion matrix ({best_id}, test)")
plt.tight_layout(); plt.savefig(FIG / "C3_confusion_matrix.png", dpi=200); plt.show()

## 9. C5 (Opsional) — Grad-CAM pada `layer4`: sebelum vs. sesudah fine-tuning
“Sebelum” = B2 (layer4 beku, bobot ImageNet) dan “sesudah” = B3 (layer4 di-fine-tune). Keduanya memakai head yang dilatih pada data yang sama.

In [ ]:
def load_exp(exp_id):
    c = CONFIGS[exp_id]; m, _ = build_model(c["arch"], c["pretrained"], c["strategy"])
    m.load_state_dict(torch.load(CKPT / f"{exp_id}.pt", map_location="cpu")); return m.to(device).eval()

def gradcam(model, x, target=None):
    store = {}
    def hook(_, __, out): out.retain_grad(); store["a"] = out
    h = model.layer4.register_forward_hook(hook)
    x = x.clone().unsqueeze(0).to(device).requires_grad_(True)   # agar graph terbentuk walau backbone beku
    model.zero_grad(set_to_none=True)
    logits = model(x); t = int(logits.argmax(1)) if target is None else target
    logits[0, t].backward()
    a = store["a"]; w = a.grad.mean(dim=(2, 3), keepdim=True)     # [1,512,1,1] bobot per channel
    cam = F.relu((w * a).sum(1, keepdim=True))                    # [1,1,7,7]
    cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
    h.remove()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), t, logits.softmax(1)[0, t].item()

def denorm(x): return (x.permute(1, 2, 0).cpu().numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)

m_before, m_after = load_exp("B2"), load_exp("B3")
test_ds = PetDataset(paths[SPLITS["test"]], labels[SPLITS["test"]], tf_plain)
pick = np.random.RandomState(0).choice(len(test_ds), 6, replace=False)

fig, axes = plt.subplots(3, len(pick), figsize=(3.2 * len(pick), 9.5))
for j, i in enumerate(pick):
    x, y = test_ds[i]; img = denorm(x)
    axes[0, j].imshow(img); axes[0, j].set_title(f"GT: {CLASSES[y]}", fontsize=9)
    for r, (m, name) in enumerate([(m_before, "B2 (layer4 beku)"), (m_after, "B3 (layer4 fine-tuned)")], start=1):
        cam, t, p = gradcam(m, x)
        axes[r, j].imshow(img); axes[r, j].imshow(cam, cmap="jet", alpha=0.45)
        axes[r, j].set_title(f"{name}\n{CLASSES[t]} ({p:.2f})", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "C5_gradcam_before_after.png", dpi=200); plt.show()

## 10. Ekspor artefak untuk pengumpulan

In [ ]:
import shutil
open("requirements.txt", "w").write(f"torch=={torch.__version__.split('+')[0]}\ntorchvision=={torchvision.__version__.split('+')[0]}\n"
                                    "numpy\npandas\nmatplotlib\nscikit-learn\nPillow\n")
open("README.md", "w").write("""# Tugas 05 - Transfer Learning (Oxford-IIIT Pet, 10 kelas)
Jalankan notebook di Google Colab (GPU T4) dengan *Run all*. Seed=42.
Keluaran: `outputs/results.csv|json` (metrik mentah), `outputs/history_all.csv` (kurva per epoch),
`outputs/figures/`, `outputs/failure_analysis.csv`.
Eksperimen: B1 scratch, B2 feature extraction, B3 fine-tuning layer4, B4 uji augmentasi, B5 EfficientNet-B0/MobileNetV3-S.
""")
NIM_NAMA = "NIM_NamaMahasiswa"   # <-- ganti
CKPT_BAK = OUT / "checkpoints"
shutil.make_archive(f"Tugas05_{NIM_NAMA}_outputs", "zip", OUT)
print("ZIP dibuat. Tambahkan notebook (.ipynb) + laporan, lalu zip ulang sesuai format Tugas05_<NIM>_<Nama>.zip")
try:
    from google.colab import files; files.download(f"Tugas05_{NIM_NAMA}_outputs.zip")
except ImportError:
    pass